# Tutorial 03: Value Learning

**Time**: 45 minutes | **Difficulty**: Beginner-Intermediate | **Prerequisites**: Tutorials 01-02

---

## What You'll Learn
- What a value function actually is
- How the Bellman equation propagates rewards backwards
- Why Q-learning converges to the optimal policy
- The difference between on-policy (SARSA) and off-policy (Q-learning)

---

## The Core Idea: What Is a Value?

A **value** answers the question: *"How good is it to be in this state (or take this action)?"*

More precisely: the expected total future reward from here, following the best possible strategy.

```
V(state) = expected total reward if I act optimally from here onward
Q(state, action) = expected total reward if I take this action, then act optimally
```

### The Bellman Equation

Values propagate backwards from the reward:

```
Q(s, a) = reward + gamma * max_a' Q(next_state, a')
```

- `reward`: immediate feedback for taking action `a` in state `s`
- `gamma`: discount factor (0-1) — how much future rewards matter
- `max_a' Q(next_state, a')`: the best possible future from next state

**Intuition**: the value of a state is the immediate reward plus the discounted value of where you end up.

---

## Setup

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout + result.stderr
    print(output if output.strip() else '(no output)')
    return result.returncode == 0

run('multiverse doctor')

## Part 1: Watch Q-Values Learn on Line World

Line World is perfect for understanding value learning because it's one-dimensional.
The optimal policy is simply: always go right.
We can watch the Q-values propagate from the goal backwards.

In [ ]:
# Short training — agent has barely learned anything yet
print('Early training (10 episodes):')
run('multiverse train --algo q --verse line_world --episodes 10 --seed 1')

In [ ]:
# Medium training — values are starting to propagate
print('Mid training (50 episodes):')
run('multiverse train --algo q --verse line_world --episodes 50 --seed 1')

In [ ]:
# Converged training — Q-values have stabilized
print('Converged (200 episodes):')
run('multiverse train --algo q --verse line_world --episodes 200 --seed 1')

## Part 2: The Discount Factor (Gamma)

Gamma controls how much the agent cares about future rewards vs immediate ones.

- **gamma = 0.0**: Agent is purely greedy — only cares about the next reward
- **gamma = 0.99**: Agent thinks long-term — future rewards barely discounted
- **gamma = 0.9**: Balanced (typical default)

In Cliff World, the short-path near the cliff has higher immediate rewards but high risk.
A high-gamma agent learns to take the safer long path.

In [ ]:
# Low gamma: short-sighted agent on cliff world
print('Low gamma (short-sighted):')
run('multiverse train --algo q --verse cliff_world --episodes 200 --seed 42 -- --gamma 0.5')

In [ ]:
# High gamma: long-sighted agent on cliff world
print('High gamma (long-sighted):')
run('multiverse train --algo q --verse cliff_world --episodes 200 --seed 42 -- --gamma 0.99')

## Part 3: Q-Learning vs SARSA

Both learn Q-values, but they differ in **which future Q-value they use for updates**:

```
Q-Learning (off-policy):
  Q(s,a) += alpha * [r + gamma * max_a' Q(s',a') - Q(s,a)]
                              ^^^^^^^^^
                    Always uses the BEST next action

SARSA (on-policy):
  Q(s,a) += alpha * [r + gamma * Q(s', a_taken) - Q(s,a)]
                              ^^^^^^^^^
                    Uses the ACTUAL next action taken
```

On cliff world, this matters a lot:
- Q-learning learns an "optimal but risky" policy (near the cliff)
- SARSA learns a "safe but suboptimal" policy (away from the cliff)

This is because SARSA accounts for its own exploratory mistakes during learning.

In [ ]:
print('Q-Learning on cliff_world:')
run('multiverse train --algo q --verse cliff_world --episodes 300 --seed 7')

In [ ]:
print('SARSA on cliff_world:')
run('multiverse train --algo sarsa --verse cliff_world --episodes 300 --seed 7')

**Look at the average returns**: SARSA likely scores lower during training (it falls off the cliff less but takes a longer path), while Q-learning's final policy is more optimal but had higher variance during learning.

Neither is universally better — choose based on whether you can afford mistakes during training.

## Part 4: Value Learning on a Harder Environment

Let's apply the same algorithms to a harder environment to see how value learning scales.

In [ ]:
# Grid world requires learning values for a 2D state space
print('Q-Learning on grid_world (harder):')
run('multiverse train --algo q --verse grid_world --episodes 300 --seed 42')

In [ ]:
print('SARSA on grid_world:')
run('multiverse train --algo sarsa --verse grid_world --episodes 300 --seed 42')

## What You Learned

| Concept | What It Means |
|---|---|
| Q-value | Expected future reward for (state, action) pair |
| Bellman equation | Q-values bootstrap from each other, propagating from reward |
| Gamma | Controls short-term vs long-term thinking |
| Q-Learning | Off-policy: learns optimal Q regardless of actual behavior |
| SARSA | On-policy: learns Q for the actual policy being followed |

Q-learning and SARSA work great for small state spaces (discrete positions, directions).
For large/continuous state spaces you need **function approximation** — that's Tutorial 04.

---

## Next

- [Tutorial 04: Policy Optimization — PPO and policy gradients](04_policy_optimization.ipynb)
- [RL Concepts Reference](../RL_CONCEPTS.md)